# Street Scout — LoRA Training on Colab T4

This notebook fine-tunes the CLIP ViT-L/14 vision encoder on your car reference images
using LoRA and Supervised Contrastive Loss.

**Before running:** upload the `street_scout` folder to your Google Drive root.
It should contain:
- `reference_images.zip` (zipped reference images)
- `class_labels.json`
- `reference_images.json`
- `train_lora.py`

Use **Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# Step 1 — check we have a GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Step 2 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3 — install peft, remove incompatible torchao, then restart runtime
# After this cell runs, Colab will restart. Re-run from Step 4 downward.
!pip install -q peft
!pip uninstall -q -y torchao
import os
os.kill(os.getpid(), 9)  # forces a clean runtime restart

In [ ]:
# Step 4 — unzip reference images (~10k images, takes a couple of minutes)
import zipfile, os

DRIVE_DIR = '/content/drive/MyDrive/street_scout'
IMAGES_DIR = '/content/reference_images'

os.makedirs(IMAGES_DIR, exist_ok=True)
zip_path = f'{DRIVE_DIR}/reference_images.zip'

print('Extracting images...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(IMAGES_DIR)

count = len([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])
print(f'Extracted {count} images to {IMAGES_DIR}')

In [ ]:
# Step 5 — copy JSON files and fix Windows paths → Colab paths
import json, shutil
from pathlib import PureWindowsPath

# Copy as-is first
shutil.copy(f'{DRIVE_DIR}/class_labels.json', '/content/class_labels.json')
shutil.copy(f'{DRIVE_DIR}/reference_images.json', '/content/reference_images.json')

# Rewrite Windows absolute paths to /content/reference_images/<filename>
with open('/content/reference_images.json') as f:
    ref = json.load(f)

fixed = {}
for key, paths in ref.items():
    path_list = paths if isinstance(paths, list) else [paths]
    fixed[key] = [f'{IMAGES_DIR}/{PureWindowsPath(p).name}' for p in path_list]

with open('/content/reference_images.json', 'w') as f:
    json.dump(fixed, f)

# Verify a couple
sample_keys = list(fixed.keys())[:2]
for k in sample_keys:
    print(k, '->', fixed[k][0])
print('Path rewrite done.')

In [ ]:
# Step 6 — copy training script to /content
shutil.copy(f'{DRIVE_DIR}/train_lora.py', '/content/train_lora.py')
print('train_lora.py ready.')

In [ ]:
# Step 7 — train  (T4 GPU, ~20-30 min for 40 epochs)
# Mixed-precision (AMP) is enabled automatically on CUDA — keeps VRAM under 6 GB.
# Best weights are saved to /content/clip_lora/ automatically.
!cd /content && python train_lora.py \
    --labels-path /content/class_labels.json \
    --ref-map     /content/reference_images.json \
    --output-dir  /content/clip_lora \
    --epochs 40 --rank 16 --p 8 --k 4

In [ ]:
# Step 8 — copy trained LoRA weights back to Google Drive for download
import shutil, os

OUTPUT_DRIVE = f'{DRIVE_DIR}/clip_lora'
os.makedirs(OUTPUT_DRIVE, exist_ok=True)

if os.path.exists('/content/clip_lora'):
    for fname in os.listdir('/content/clip_lora'):
        shutil.copy(f'/content/clip_lora/{fname}', f'{OUTPUT_DRIVE}/{fname}')
    print('LoRA weights saved to Drive:', OUTPUT_DRIVE)
    print('Files:', os.listdir(OUTPUT_DRIVE))
else:
    print('ERROR: /content/clip_lora not found — training may have failed.')

## After training

1. Download the `clip_lora` folder from Google Drive (right-click → Download).
2. Place it at: `ai-service/model/clip_lora/` on your PC.
3. Delete the embedding cache: `ai-service/model/reference_embeddings.pt`
4. Restart the ai-service: `uvicorn src.main:app --reload --port 8000`

The service will load the LoRA adapter automatically on next startup.